# G-Memory Poster Demo

This notebook is the self-contained front end for the course-project demo layer. It wraps the official G-Memory repository, keeps the task small enough for a laptop, and exposes the retrieval pipeline in a way that is easy to inspect. The full repository with scripts, cached outputs, and documentation is here: [hku-comp7404-gmemory-poster-demo](https://github.com/nZiben/hku-comp7404-gmemory-poster-demo).

## 1. Project Overview

- The official repository is reused from `official_gmemory/`.
- The demo layer adds small-task selection, structured tracing, visualization, and replay.
- The recommended live path is a local `pddl` gripper example with `autogen` agents.

## 2. What G-Memory Adds Over Standard MAS

G-Memory is not just a larger context window. It builds a hierarchy of past tasks, distilled insights, and interaction trajectories, then injects targeted memory back into the multi-agent workflow.

In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'demo' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from demo.utils.official_wrapper import discover_available_tasks, recommended_task_name
from demo.utils.memory_introspection import extract_case_view, build_compare_payload
from demo.utils.visualize_graphs import save_memory_hierarchy_figure, save_architecture_overview
from demo.utils.visualize_compare import save_compare_figure
from demo.utils.caching import latest_json


## 3. Environment / Repo Sanity Checks

In [2]:
available = discover_available_tasks()
available

{'alfworld': {'available': True,
  'count': 134,
  'path': '/Users/sergey/code/comp7404-gmemory-poster-demo/official_gmemory/data/alfworld/alfworld_tasks_suffix.json'},
 'fever': {'available': True,
  'count': 9999,
  'path': '/Users/sergey/code/comp7404-gmemory-poster-demo/official_gmemory/data/fever/fever_dev.jsonl'},
 'pddl': {'available': True,
  'count': 60,
  'path': '/Users/sergey/code/comp7404-gmemory-poster-demo/official_gmemory/data/pddl/test.jsonl'},
 'sciworld': {'available': True,
  'count': 90,
  'path': '/Users/sergey/code/comp7404-gmemory-poster-demo/official_gmemory/data/sciworld/test.jsonl'}}

## 4. Detect Available Tasks and Configs

In [3]:
recommended_task_name()

'pddl'

## 5. Select the Recommended Demo Task Automatically

The wrapper prefers `pddl` when it is present, because it is fully local and does not require external search.

## 6. Run or Load the No-Memory Baseline

In [4]:
compare_json = latest_json(PROJECT_ROOT / 'demo/cached_runs')
assert compare_json is not None, 'Run the live compare once or add a cached compare JSON first.'
compare_record = json.loads(compare_json.read_text(encoding='utf-8'))
no_memory_view = extract_case_view(compare_record['baseline_case'])
no_memory_view

{'query': 'The goal is to satisfy the following conditions: ball1 is at roomb. , ball2 is at roomb. , ball3 is at roomb. , ball4 is at roomb. ',
 'task_description': 'Here is your initial observation: Ball1 is a ball.  Ball1 is at rooma.  Ball2 is a ball.  Ball2 is at rooma.  Ball3 is a ball.  Ball3 is at rooma.  Ball4 is a ball.  Ball4 is at rooma.  Left is a gripper.  Left is free.  Right is a gripper.  Right is free.  Robby is at rooma.  Room rooma Room roomb\n**Here is your task: The goal is to satisfy the following conditions: ball1 is at roomb. , ball2 is at roomb. , ball3 is at roomb. , ball4 is at roomb. ',
 'retrieved_historical_queries': ['not exposed by current code path'],
 'selected_successful_trajectories': ['not exposed by current code path'],
 'selected_failed_trajectories': [],
 'selected_insights': ['not exposed by current code path'],
 'agent_memory_packets': {'solver': {'role': 'solver',
   'successful_reference_cases': [],
   'insights': []},
  'ground_truth': {'ro

## 7. Run or Load the G-Memory Version

In [5]:
gmemory_view = extract_case_view(compare_record['gmemory_case'])
gmemory_view

{'query': 'The goal is to satisfy the following conditions: ball1 is at roomb. , ball2 is at roomb. , ball3 is at roomb. , ball4 is at roomb. ',
 'task_description': 'Here is your initial observation: Ball1 is a ball.  Ball1 is at rooma.  Ball2 is a ball.  Ball2 is at rooma.  Ball3 is a ball.  Ball3 is at rooma.  Ball4 is a ball.  Ball4 is at rooma.  Left is a gripper.  Left is free.  Right is a gripper.  Right is free.  Robby is at rooma.  Room rooma Room roomb\n**Here is your task: The goal is to satisfy the following conditions: ball1 is at roomb. , ball2 is at roomb. , ball3 is at roomb. , ball4 is at roomb. ',
 'retrieved_historical_queries': ['The goal is to satisfy the following conditions: ball1 is at roomb. , ball2 is at roomb. , ball3 is at roomb. , ball4 is at roomb. '],
 'selected_successful_trajectories': [{'task_main': 'The goal is to satisfy the following conditions: ball1 is at roomb. , ball2 is at roomb. , ball3 is at roomb. , ball4 is at roomb. ',
   'task_description

## 8. Inspect Retrieved Memory

In [6]:
compare_payload = compare_record['compare_payload']
{
    'retrieved_historical_queries': gmemory_view['retrieved_historical_queries'],
    'selected_successful_trajectories': gmemory_view['selected_successful_trajectories'],
    'selected_insights': gmemory_view['selected_insights'],
}

{'retrieved_historical_queries': ['The goal is to satisfy the following conditions: ball1 is at roomb. , ball2 is at roomb. , ball3 is at roomb. , ball4 is at roomb. '],
 'selected_successful_trajectories': [{'task_main': 'The goal is to satisfy the following conditions: ball1 is at roomb. , ball2 is at roomb. , ball3 is at roomb. , ball4 is at roomb. ',
   'task_description': 'Here is your initial observation: Ball1 is a ball.  Ball1 is at rooma.  Ball2 is a ball.  Ball2 is at rooma.  Ball3 is a ball.  Ball3 is at rooma.  Ball4 is a ball.  Ball4 is at rooma.  Left is a gripper.  Left is free.  Right is a gripper.  Right is free.  Robby is at rooma.  Room rooma Room roomb\n**Here is your task: The goal is to satisfy the following conditions: ball1 is at roomb. , ball2 is at roomb. , ball3 is at roomb. , ball4 is at roomb. \n- Environment feedback\n\n',
   'task_trajectory': '> pick ball1 rooma right\nBall1 is a ball.  Ball1 is carrying right.  Ball2 is a ball.  Ball2 is at rooma.  Ball

## 9. Visualize Query / Insight / Interaction Structures

In [7]:
memory_fig_path = PROJECT_ROOT / 'demo/outputs/notebook_memory_flow.png'
save_memory_hierarchy_figure(compare_payload, memory_fig_path)
memory_fig_path

PosixPath('/Users/sergey/code/comp7404-gmemory-poster-demo/demo/outputs/notebook_memory_flow.png')

## 10. Show Per-Agent Memory Assignment

In [8]:
gmemory_view['agent_memory_packets']

{'solver': {'role': 'solver',
  'successful_reference_cases': ['\n### Task description:   \nHere is your initial observation: Ball1 is a ball.  Ball1 is at rooma.  Ball2 is a ball.  Ball2 is at rooma.  Ball3 is a ball.  Ball3 is at rooma.  Ball4 is a ball.  Ball4 is at rooma.  Left is a gripper.  Left is free.  Right is a gripper.  Right is free.  Robby is at rooma.  Room rooma Room roomb\n**Here is your task: The goal is to satisfy the following conditions: ball1 is at roomb. , ball2 is at roomb. , ball3 is at roomb. , ball4 is at roomb. \n- Environment feedback\n\n    \n\n### Key steps:\n1. Identify which balls still need to reach the target room.\n2. Pick up one or two balls in rooma using the available grippers.\n3. Move from rooma to roomb only after the intended pickup batch is ready.\n4. Drop the carried balls in roomb and repeat until all goal conditions are satisfied.\n\n### Detailed trajectory:\n> pick ball1 rooma right\nBall1 is a ball.  Ball1 is carrying right.  Ball2 is a 

## 11. Compare Final Outputs

In [9]:
compare_fig_path = PROJECT_ROOT / 'demo/outputs/notebook_compare.png'
save_compare_figure(compare_payload, compare_fig_path)
compare_fig_path

PosixPath('/Users/sergey/code/comp7404-gmemory-poster-demo/demo/outputs/notebook_compare.png')

## 12. Summarize Why the Behavior Changed

In [10]:
compare_payload['delta_summary']

'G-Memory succeeds where the empty-memory baseline fails. It retrieves 1 related historical query nodes and 2 insight nodes, then injects role-specific memory packets before action selection.'

## 13. Replay Cached Example

In [11]:
arch_fig_path = PROJECT_ROOT / 'demo/outputs/notebook_architecture.png'
save_architecture_overview(arch_fig_path)
arch_fig_path

PosixPath('/Users/sergey/code/comp7404-gmemory-poster-demo/demo/outputs/notebook_architecture.png')

## 14. Final Takeaway

The important point is not the absolute score of this tiny local run. It is that the official pipeline is being reused, the retrieval path is visible, and the demo includes both a live mode and a replay-safe fallback.